### Load processed data

In [1]:
import geopandas as gpd
from pathlib import Path

fire_data_by_countries_dates = gpd.read_file(
    Path.cwd().parent.joinpath("data/processed", "fire_data_by_countries_dates.gpkg"),
)

fire_data_by_countries_dates.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   iso_code    168 non-null    str     
 1   2026-04-19  168 non-null    float64 
 2   2026-04-20  168 non-null    float64 
 3   2026-04-21  168 non-null    float64 
 4   2026-04-22  168 non-null    float64 
 5   2026-04-23  168 non-null    float64 
 6   2026-04-24  168 non-null    float64 
 7   2026-04-25  168 non-null    float64 
 8   2026-04-26  168 non-null    float64 
 9   2026-04-27  168 non-null    float64 
 10  2026-04-28  168 non-null    float64 
 11  2026-04-30  168 non-null    float64 
 12  2026-05-01  168 non-null    float64 
 13  2026-05-02  168 non-null    float64 
 14  name        168 non-null    str     
 15  geometry    168 non-null    geometry
dtypes: float64(13), geometry(1), str(2)
memory usage: 21.1 KB


### Create `output/` directory if not present yet

In [2]:
dir_path = Path(Path.cwd().parent.joinpath("output"))
if not dir_path.exists():
    Path.mkdir(dir_path)

In [ ]:
from datetime import timezone
import datetime
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


cmap = plt.colormaps["Reds"]
vmin = fire_data_by_countries_dates.iloc[:, 1:14].min().min()
vmax = fire_data_by_countries_dates.iloc[:, 1:14].max().max()


In [6]:
# assign color to value
def to_hex(val):
    rgba = cmap((val - vmin) / (vmax - vmin))
    return mcolors.to_hex(rgba)


# build the style dict
style_dict = {}

# iterate over ids of grouped country data : essentially only one id per country
counter = -1
for row in fire_data_by_countries_dates.itertuples():
    # each feature gets its own dict of timestamps
    print(row)
    ts_dict = {}
    for i in range(2, 15):
        color = to_hex(float(f"{row[i]:.3f}"))
        # index of rows values is shifted compared to columns index
        timestr = fire_data_by_countries_dates.columns[i - 1]
        epoch_t = (
            (datetime.datetime.strptime(timestr, "%Y-%m-%d"))
            .replace(tzinfo=timezone.utc)
            .timestamp()
        )

        # for some reasons folium refuses to recognize a unix timestamp with one decimal as the as the same of one without decimal digits,
        # even though the converted date is exactly the same, and it will then refuse to show any geometry for any day in the map.
        ts_dict[f"{epoch_t:.0f}"] = {
            "color": color,
            "opacity": 1,
        }

    style_dict[str(getattr(row, "Index"))] = ts_dict

Pandas(Index=0, iso_code='AFG', _2=4.870801231485595e-05, _3=6.227886102899504e-05, _4=0.0, _5=0.0, _6=0.0, _7=0.00010587100035229066, _8=0.00012768238699894315, _9=2.323586625208694e-05, _10=0.0, _11=0.0, _12=6.396372937950894e-05, _13=0.0, _14=0.0, name='Afghanistan', geometry=<MULTIPOLYGON (((74.89 37.234, 74.804 37.355, 74.706 37.384, 74.566 37.372, ...>)
Pandas(Index=1, iso_code='AGO', _2=0.0009448544156573354, _3=0.0014124488650036097, _4=0.0012669768188016363, _5=0.0011877837490976177, _6=0.0003540306408919548, _7=0.00020608807251143018, _8=0.0004827464506296623, _9=0.0006642576401700489, _10=0.00125435148792813, _11=0.0, _12=0.0, _13=0.0006118713403384935, _14=0.001142071067618513, name='Angola', geometry=<MULTIPOLYGON (((11.716 -16.508, 11.67 -16.543, 11.736 -16.699, 11.716 -16.5...>)
Pandas(Index=2, iso_code='ALB', _2=0.00223895652173913, _3=0.0005683478260869565, _4=0.0, _5=0.0, _6=0.0, _7=0.0005683478260869565, _8=0.0, _9=0.001492869565217391, _10=0.001408, _11=0.0005683478

In [7]:
style_dict

{'0': {'1776556800': {'color': '#fff5f0', 'opacity': 1},
  '1776643200': {'color': '#fff5f0', 'opacity': 1},
  '1776729600': {'color': '#fff5f0', 'opacity': 1},
  '1776816000': {'color': '#fff5f0', 'opacity': 1},
  '1776902400': {'color': '#fff5f0', 'opacity': 1},
  '1776988800': {'color': '#fff5f0', 'opacity': 1},
  '1777075200': {'color': '#fff5f0', 'opacity': 1},
  '1777161600': {'color': '#fff5f0', 'opacity': 1},
  '1777248000': {'color': '#fff5f0', 'opacity': 1},
  '1777334400': {'color': '#fff5f0', 'opacity': 1},
  '1777507200': {'color': '#fff5f0', 'opacity': 1},
  '1777593600': {'color': '#fff5f0', 'opacity': 1},
  '1777680000': {'color': '#fff5f0', 'opacity': 1}},
 '1': {'1776556800': {'color': '#fff5f0', 'opacity': 1},
  '1776643200': {'color': '#fff5f0', 'opacity': 1},
  '1776729600': {'color': '#fff5f0', 'opacity': 1},
  '1776816000': {'color': '#fff5f0', 'opacity': 1},
  '1776902400': {'color': '#fff5f0', 'opacity': 1},
  '1776988800': {'color': '#fff5f0', 'opacity': 1},
 

In [8]:
geojson_str = fire_data_by_countries_dates.to_json()

In [10]:
import folium
from folium.plugins import TimeSliderChoropleth

# initialise a world view
m = folium.Map(location=[0, 0], zoom_start=2, tiles="CartoDB positron")

# add the timeslider choropleth
TimeSliderChoropleth(
    data=geojson_str,
    styledict=style_dict,
    name="Fire area over time",
    # optional: start the slider at the most recent date
    init_timestamp=-1,
).add_to(m)

folium.LayerControl().add_to(m)

# display in a Jupyter notebook or save to HTML
m.save(Path.cwd().parent.joinpath("output", "map_time_slider.html"))